In [90]:
!pip install statsmodels

In [91]:
import pyodbc
import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import matplotlib.pyplot as plt

In [92]:
conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=LAPTOP-I4NKHGBJ\\SQLEXPRESS;"
    "DATABASE=QLPKS;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()

print("Kết nối thành công")

Kết nối thành công


In [93]:
query = """
SELECT
    Ngay AS ds,
    TyLeCongSuat AS y
FROM CongSuatPhong
ORDER BY Ngay
"""

df = pd.read_sql(query, conn)

print(df.head())
print(df.shape)

           ds      y
0  2015-07-01  40.39
1  2015-07-02  14.12
2  2015-07-03  14.51
3  2015-07-04  17.65
4  2015-07-05  14.51
(793, 2)


C:\Users\PC\AppData\Local\Temp\ipykernel_4560\1252469838.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [94]:
print(df.shape)
df.describe()

(793, 2)


,y
count,793.000000
mean,37.171311
std,13.519610
min,5.880000
25%,27.450000
50%,36.470000
75%,45.880000
max,100.000000


In [95]:
#Chia Train/Test
train_size = int(len(df) * 0.8)

train = df[:train_size]

test = df[train_size:]

print(len(train))
print(len(test))

634
159


In [96]:
#Huấn luyện Prophet
model_test = Prophet()

model_test.fit(train)

21:28:13 - cmdstanpy - INFO - Chain [1] start processing
21:28:13 - cmdstanpy - INFO - Chain [1] done processing


In [97]:
#Dự báo tập test
future_test = model_test.make_future_dataframe(
    periods=len(test)
)

forecast_test = model_test.predict(
    future_test
)

In [98]:
#Tính RMSE MAE MAPE
pred = forecast_test['yhat'].tail(
    len(test)
).values

actual = test['y'].values

In [99]:
mae = mean_absolute_error(
    actual,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        pred
    )
)

mape = np.mean(
    np.abs(
        (actual - pred) / actual
    )
) * 100

In [100]:
print("RMSE =", rmse)
print("MAE =", mae)
print("MAPE =", mape)

RMSE = 10.720643081506
MAE = 8.109410405338119
MAPE = 17.724874368331918


In [101]:
# ARIMA

arima_model = ARIMA(
    train['y'],
    order=(5,1,0)
)

arima_model_fit = arima_model.fit()

pred_arima = arima_model_fit.forecast(
    steps=len(test)
)

In [102]:
#Tính RMSE, MAE, MAPE cho ARIMA
rmse_arima = np.sqrt(
    mean_squared_error(
        test['y'],
        pred_arima
    )
)

mae_arima = mean_absolute_error(
    test['y'],
    pred_arima
)

mape_arima = np.mean(
    np.abs(
        (test['y'] - pred_arima)
        / test['y']
    )
) * 100

In [103]:
print(df['y'].describe())

count    793.000000
mean      37.171311
std       13.519610
min        5.880000
25%       27.450000
50%       36.470000
75%       45.880000
max      100.000000
Name: y, dtype: float64


In [104]:
#So sánh kết quả 2 mô hình
print("===== PROPHET =====")
print("RMSE =", rmse)
print("MAE  =", mae)
print("MAPE =", mape)

print()

print("===== ARIMA =====")
print("RMSE =", rmse_arima)
print("MAE  =", mae_arima)
print("MAPE =", mape_arima)

===== PROPHET =====
RMSE = 10.720643081506
MAE  = 8.109410405338119
MAPE = 17.724874368331918

===== ARIMA =====
RMSE = 9.411881602708053
MAE  = 7.037709825265359
MAPE = 15.849360973000461


In [105]:
#Chọn mô hình tốt nhất
if rmse < rmse_arima:
    best_model = "Prophet"
else:
    best_model = "ARIMA"

print("Mo hinh tot nhat:", best_model)

Mo hinh tot nhat: ARIMA


In [106]:
#Lưu mô hình tốt nhất
cursor = conn.cursor()

cursor.execute("""
INSERT INTO DanhGiaMoHinh
(
    TenMoHinh,
    RMSE,
    MAE,
    MAPE
)
VALUES (?, ?, ?, ?)
""",
"Prophet",
float(rmse),
float(mae),
float(mape)
)

cursor.execute("""
INSERT INTO DanhGiaMoHinh
(
    TenMoHinh,
    RMSE,
    MAE,
    MAPE
)
VALUES (?, ?, ?, ?)
""",
"ARIMA",
float(rmse_arima),
float(mae_arima),
float(mape_arima)
)

conn.commit()

In [107]:
#Huấn luyện lại toàn bộ dữ liệu
if best_model == "Prophet":

    model = Prophet()

    model.fit(df)

    future = model.make_future_dataframe(
        periods=30
    )

    forecast = model.predict(future)

    forecast_30 = forecast.tail(30)

else:

    model = ARIMA(
        df['y'],
        order=(5,1,0)
    )

    model_fit = model.fit()

    future_dates = pd.date_range(
        start=df['ds'].max(),
        periods=31,
        freq='D'
    )[1:]

    pred = model_fit.forecast(
        steps=30
    )

    forecast_30 = pd.DataFrame({
        'ds': future_dates,
        'yhat': pred
    })

In [108]:
#Dự báo 30 ngày
if best_model == "Prophet":

    future = model.make_future_dataframe(
        periods=30
    )

    forecast = model.predict(
        future
    )

    forecast_30 = forecast.tail(30)

else:

    model = ARIMA(
        df['y'],
        order=(5,1,0)
    )

    model_fit = model.fit()

    future_dates = pd.date_range(
        start=df['ds'].max(),
        periods=31,
        freq='D'
    )[1:]

    pred = model_fit.forecast(
        steps=30
    )

    forecast_30 = pd.DataFrame({
        'ds': future_dates,
        'yhat': pred
    })

In [109]:
forecast_30.head()

,ds,yhat
793,2017-09-01,39.637929
794,2017-09-02,36.777021
795,2017-09-03,35.673032
796,2017-09-04,34.663088
797,2017-09-05,35.342092


In [110]:
#Công suất trung bình dự báo
avg_occ = float(
    forecast_30['yhat'].mean()
)

print(avg_occ)

35.863460079456985


In [111]:
#Độ tin cậy
do_tin_cay = 100 - mape

print("Do tin cay:", do_tin_cay)

Do tin cay: 82.27512563166809


In [112]:
#Lưu vào bảng DuBao
cursor.execute("""
INSERT INTO DuBao
(
    SoNgayDuBao,
    CongSuatTrungBinh,
    MoHinhSuDung,
    DoTinCay
)
VALUES
(
    ?, ?, ?, ?
)
""",
30,
avg_occ,
best_model,
float(do_tin_cay)
)

conn.commit()

In [113]:
#Lấy MaDuBao mới nhất
cursor.execute("""
SELECT MAX(MaDuBao)
FROM DuBao
""")

MaDuBao = cursor.fetchone()[0]

print(MaDuBao)

7


In [114]:
#Lưu ChiTietDuBao
for _, row in forecast_30.iterrows():

    cursor.execute("""
    INSERT INTO ChiTietDuBao
    (
        MaDuBao,
        Ngay,
        CongSuatDuBao
    )
    VALUES
    (
        ?, ?, ?
    )
    """,
    MaDuBao,
    row['ds'],
    float(row['yhat'])
    )

conn.commit()

In [115]:
#Lưu NhatKyDuBao
cursor.execute("""
INSERT INTO NhatKyDuBao
(
    MoHinh,
    SoNgayDuBao,
    RMSE,
    MAE,
    MAPE,
    GhiChu
)
VALUES
(
    ?, ?, ?, ?, ?, ?
)
""",
best_model,
30,
float(rmse),
float(mae),
float(mape),
f'Du bao tu dong bang {best_model}'
)

conn.commit()

In [116]:
#Sinh KhuyenNghi
if avg_occ >= 80:

    recommendation = (
        "Cong suat phong cao, can bo sung nhan su"
    )

elif avg_occ <= 40:

    recommendation = (
        "Cong suat phong thap, nen trien khai khuyen mai"
    )

else:

    recommendation = (
        "Cong suat phong on dinh"
    )

cursor.execute("""
INSERT INTO KhuyenNghi
(
    MaDuBao,
    NoiDung
)
VALUES
(
    ?, ?
)
""",
MaDuBao,
recommendation
)

conn.commit()

In [117]:
#Sinh QuyetDinh
if avg_occ >= 80:

    loai = 'Tang nhan su'

    noidung = (
        'Bo sung nhan vien le tan va don phong'
    )

elif avg_occ <= 40:

    loai = 'Khuyen mai'

    noidung = (
        'Giam gia phong trong mua thap diem'
    )

else:

    loai = 'Duy tri'

    noidung = (
        'Van hanh binh thuong'
    )

cursor.execute("""
INSERT INTO QuyetDinh
(
    MaDuBao,
    LoaiQuyetDinh,
    NoiDung
)
VALUES
(
    ?, ?, ?
)
""",
MaDuBao,
loai,
noidung
)

conn.commit()

In [118]:
#Sinh CanhBao
if avg_occ >= 80:

    mucdo = 'Cao'

elif avg_occ <= 40:

    mucdo = 'Thap'

else:

    mucdo = 'Trung binh'

cursor.execute("""
INSERT INTO CanhBao
(
    NgayCanhBao,
    MucDo,
    NoiDung
)
VALUES
(
    GETDATE(),
    ?,
    ?
)
""",
mucdo,
recommendation
)

conn.commit()

print("Hoan thanh DSS")

Hoan thanh DSS
